# Import all needed Modules and set constants (copied from course)

## Sizes
64 was the base in the course. I want to go as low as 16 since at least humans can still recognize patterns in colored images with a size of 4 by 4 (Check out https://de.sporcle.com/games/ohicanchat/name-the-league-of-legends-characters-from-16-pixels if you know League of Legends and want a challenge) Since our AI has to work on grayscale (I think) this is a bit harsh, but maybe 16 by 16 will yield results.

In [12]:
import cv2
import json
from matplotlib import pyplot as plt
import numpy as np
import os
import random

# import a lot of things from keras:
# sequential model
from keras.models import Sequential

# layers
from keras.layers import Input, Dense, Dropout, Flatten, Conv2D, MaxPooling2D, RandomFlip, RandomRotation, RandomContrast, RandomBrightness

# loss function
from keras.metrics import categorical_crossentropy

# callback functions
from keras.callbacks import ReduceLROnPlateau, EarlyStopping

# convert data to categorial vector representation
from keras.utils import to_categorical

# nice progress bar for loading data
from tqdm import tqdm

# helper function for train/test split
from sklearn.model_selection import train_test_split

# import confusion matrix helper function
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# import pre-trained model
from keras.applications.vgg16 import VGG16

# include only those gestures
CONDITIONS = ['like', 'stop']

# image size
# Use a broad array of selections
# Is 16 even viable? We  will find out!
# I really want to do 512 as well, but I will first see how long training takes on 256
IMG_SIZES = [16, 32, 64, 128, 256]


PATH = '../dataset_sample/'

# number of color channels we want to use
# set to 1 to convert to grayscale
# set to 3 to use color images
COLOR_CHANNELS = 3

## helper function to load and parse annotations (copied from course)


In [13]:
annotations = dict()

for condition in CONDITIONS:
    with open(f'{PATH}_annotations/{condition}.json') as f:
        annotations[condition] = json.load(f)

## helper function to pre-process images (color channel conversion and resizing) (also copied from course and adjusted slightly to take size as an arg)

In [14]:
def preprocess_image(img, size):
    if COLOR_CHANNELS == 1:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_resized = cv2.resize(img, (size, size)) # Use this tuple directly to make it loopable better later
    return img_resized

## Load images and annotations. Taken from the course and then adjusted because we of course need 5 different lists this time

In [15]:
# We can use dicts and store lists using the size as a key
images = {} # stores actual image data
for size in IMG_SIZES:
    images[size] = []
labels = {} # stores labels (as integer - because this is what our network needs)
for size in IMG_SIZES:
    labels[size] = []
label_names = {} # maps label ints to their actual categories so we can understand predictions later
for size in IMG_SIZES:
    label_names[size] = []

# loop over all conditions
# loop over all files in the condition's directory
# read the image and corresponding annotation
# crop image to the region of interest
# preprocess image
# store preprocessed image and label in corresponding lists
for condition in CONDITIONS:
    for filename in tqdm(os.listdir(f'{PATH}/{condition}')):
        # extract unique ID from file name
        UID = filename.split('.')[0]
        img = cv2.imread(f'{PATH}/{condition}/{filename}')

        # get annotation from the dict we loaded earlier
        try:
            annotation = annotations[condition][UID]
        except Exception as e:
            print(e)
            continue

        # iterate over all hands annotated in the image
        for i, bbox in enumerate(annotation['bboxes']):
            # annotated bounding boxes are in the range from 0 to 1
            # therefore we have to scale them to the image size
            x1 = int(bbox[0] * img.shape[1])
            y1 = int(bbox[1] * img.shape[0])
            w = int(bbox[2] * img.shape[1])
            h = int(bbox[3] * img.shape[0])
            x2 = x1 + w
            y2 = y1 + h

            # crop image to the bounding box and apply pre-processing
            crop = img[y1:y2, x1:x2]

            # We need to repeat this part for every size this time unlike in the course where we only did it once
            for size in IMG_SIZES:

                preprocessed = preprocess_image(crop, size)

                # get the annotated hand's label
                # if we have not seen this label yet, add it to the list of labels
                label = annotation['labels'][i]
                if label == "no_gesture":
                    continue
                if label not in label_names[size]:
                    label_names[size].append(label)

                label_index = label_names[size].index(label)

                images[size].append(preprocessed)
                labels[size].append(label_index)





  0%|          | 0/250 [00:00<?, ?it/s]



  1%|          | 3/250 [00:00<00:12, 19.11it/s]



  2%|▏         | 6/250 [00:00<00:10, 23.70it/s]



  4%|▎         | 9/250 [00:00<00:11, 21.68it/s]



  5%|▍         | 12/250 [00:00<00:10, 21.68it/s]



  6%|▌         | 15/250 [00:00<00:10, 22.71it/s]



  7%|▋         | 18/250 [00:00<00:10, 23.09it/s]



  8%|▊         | 21/250 [00:00<00:09, 23.13it/s]



 10%|▉         | 24/250 [00:01<00:09, 22.61it/s]



 11%|█         | 27/250 [00:01<00:09, 22.38it/s]



 12%|█▏        | 30/250 [00:01<00:09, 22.40it/s]



 14%|█▎        | 34/250 [00:01<00:08, 24.78it/s]



 15%|█▍        | 37/250 [00:01<00:10, 20.89it/s]



 16%|█▌        | 40/250 [00:01<00:09, 21.28it/s]



 17%|█▋        | 43/250 [00:01<00:09, 21.57it/s]



 18%|█▊        | 46/250 [00:02<00:09, 21.59it/s]



 20%|█▉        | 49/250 [00:02<00:08, 22.98it/s]



 21%|██        | 52/250 [00:02<00:08, 22.58it/s]



 22%|██▏       | 56/250 [00:02<00:07, 25.59it/s]



 24%|██▎       | 59/

## split data set into train and test

x is for the actual data, y is for the label (this is convention). We copy this from the course and run it for each image size

In [16]:
X_train = {}
X_test = {}
y_train = {}
y_test = {}
for size in IMG_SIZES:
    X_train_subset, X_test_subset, y_train_subset, y_test_subset = train_test_split(images[size], labels[size], test_size=0.2, random_state=42)
    X_train[size] = X_train_subset
    X_test[size] = X_test_subset
    y_train[size] = y_train_subset
    y_test[size] = y_test_subset


## transform data sets into a format compatible with our neural network

image data has to be a numpy array with following dimensions: [image_id, y_axis, x_axis, color_channels]

furthermore, scale all values to a range of 0 to 1

training data has to be converted to a categorial vector ("one hot"):

[3] --> [0, 0, 0, 1, 0, ..., 0]

This is also copied from the course script and then adjusted to run 5 times

In [18]:
train_label = {}
test_label = {}

for size in IMG_SIZES:
    X_train[size] = np.array(X_train[size]).astype('float32')
    X_train[size] /= 255

    X_test[size] = np.array(X_test[size]).astype('float32')
    X_test[size] /= 255

    y_train_one_hot = to_categorical(y_train[size])
    y_test_one_hot = to_categorical(y_test[size])

    train_label[size] = y_train_one_hot
    test_label[size] = y_test_one_hot

    X_train[size] = X_train[size].reshape(-1, size, size, COLOR_CHANNELS)
    X_test[size] = X_test[size].reshape(-1, size, size, COLOR_CHANNELS)



## Adding a check if everything went right

In [19]:
for size in IMG_SIZES:
    print(
        size,
        X_train[size].shape,
        X_test[size].shape,
        train_label[size].shape,
        test_label[size].shape
    )

16 (400, 16, 16, 3) (100, 16, 16, 3) (400, 2) (100, 2)
32 (400, 32, 32, 3) (100, 32, 32, 3) (400, 2) (100, 2)
64 (400, 64, 64, 3) (100, 64, 64, 3) (400, 2) (100, 2)
128 (400, 128, 128, 3) (100, 128, 128, 3) (400, 2) (100, 2)
256 (400, 256, 256, 3) (100, 256, 256, 3) (400, 2) (100, 2)
